# 03 · Evaluation framework and non-personalized baselines

Every later model (ItemKNN, ALS, PyTorch) plugs into the same `evaluate()` and must beat these numbers.
All tuning happens on `split_val`; `split_test` stays untouched until the end.

In [ ]:
import pandas as pd
from musicrec import config
from musicrec.preprocess import Split
from musicrec.evaluation import evaluate
from musicrec.models.baselines import Random, MostPopular

pd.set_option("display.float_format", "{:,.4f}".format)
split = Split.load(config.SPLIT_VAL_DIR)
print(f"{split.n_users} users x {split.n_items:,} items | {len(split.targets):,} target pairs")

In [ ]:
models = [
    Random(),
    MostPopular(by="plays"),
    MostPopular(by="listeners"),
    MostPopular(by="listeners", recent_days=30),
    MostPopular(by="listeners", recent_days=7),
]
results = {}
for m in models:
    results[m.name] = evaluate(m.fit(split), split)
baselines = pd.DataFrame(results).T
baselines

### Robustness: "discoveries that stuck"

75% of discoveries are played exactly once in the eval window, which could be a skip or autoplay.
Re-evaluate counting only tracks the user played at least twice. If the model ranking stays the
same, our conclusions don't depend on that noise.

In [ ]:
stuck = {m.name: evaluate(m, split, min_plays=2) for m in models}
pd.DataFrame(stuck).T[["recall@10", "ndcg@10", "recall@50", "n_eval_users"]]

In [ ]:
baselines.to_csv(config.PROJECT_ROOT / "reports" / "results_val_baselines.csv")